In [ ]:
import pandas as pd
import random
import numpy as np
from sklearn.model_selection import train_test_split

In [ ]:
final_df = pd.read_csv("finalmodel_dataset.csv")

In [ ]:
random.seed(42)
np.random.seed(42)

In [ ]:
X = final_df.drop(columns=['SK_ID_CURR', 'TARGET'])

In [ ]:
y = final_df['TARGET']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns

In [ ]:
cat_cols = X_train.select_dtypes(include=['object']).columns

In [ ]:
train_medians = X_train[num_cols].median()

In [ ]:
X_train[num_cols] = X_train[num_cols].fillna(X_train[num_cols].median())

In [ ]:
low_unique_cols = [col for col in num_cols if X_train[col].nunique() < 10]

In [ ]:
len(low_unique_cols)

In [ ]:
true_num_cols = [col for col in num_cols if X_train[col].nunique() >= 10]

In [ ]:
!pip install --upgrade pip
!pip install optbinning

In [ ]:
from optbinning import OptimalBinning

opt_num = {}

for col in true_num_cols:
    optb = OptimalBinning(
        name=col,
        dtype="numerical",
        monotonic_trend="auto",
        max_n_bins=5,
        random_state=42
    )

    optb.fit(X_train[col].values, y_train.values)
    opt_num[col] = optb

In [ ]:
opt_low = {}

for col in low_unique_cols:
    optb = OptimalBinning(
        name=col,
        dtype="categorical",
        random_state=42
    )
    optb.fit(X_train[col].values.astype(str), y_train.values)
    opt_low[col] = optb

In [ ]:
from optbinning import OptimalBinning

opt_cat = {}

for col in cat_cols:
    optb = OptimalBinning(
        name=col,
        dtype="categorical",
        random_state=42
    )

    optb.fit(X_train[col].values, y_train.values)
    opt_cat[col] = optb

In [ ]:
for col in true_num_cols:
    X_train[col] = opt_num[col].transform(X_train[col].values, metric="woe")

for col in cat_cols:
    X_train[col] = opt_cat[col].transform(X_train[col].values, metric="woe")

In [ ]:
for col in low_unique_cols:
    X_train[col] = opt_low[col].transform(X_train[col].values.astype(str), metric="woe")

In [ ]:
test_num_cols = num_cols

In [ ]:
test_cat_cols = cat_cols

In [ ]:
test_true_num_cols = true_num_cols

In [ ]:
X_test[test_num_cols] = X_test[test_num_cols].fillna(train_medians)

In [ ]:
for col in test_true_num_cols:
    X_test[col] = opt_num[col].transform(X_test[col].values, metric="woe")

In [ ]:
for col in test_cat_cols:
    X_test[col] = opt_cat[col].transform(X_test[col].values, metric="woe")

In [ ]:
for col in low_unique_cols:
    X_test[col] = opt_low[col].transform(X_test[col].values.astype(str), metric="woe")

In [ ]:
iv_values = {}

for col in true_num_cols:
    bt = opt_num[col].binning_table
    bt.build()
    iv_values[col] = bt.iv

for col in low_unique_cols:
    bt = opt_low[col].binning_table
    bt.build()
    iv_values[col] = bt.iv

for col in cat_cols:
    bt = opt_cat[col].binning_table
    bt.build()
    iv_values[col] = bt.iv

iv_df = pd.DataFrame.from_dict(iv_values, orient='index', columns=['IV'])
iv_df = iv_df.sort_values('IV', ascending=False)
pd.set_option('display.max_rows', None)
print(iv_df)

In [ ]:
selected_features = iv_df[iv_df['IV'] >= 0.02].index.tolist()
X_train = X_train[selected_features]
X_test = X_test[selected_features]

print(f"Features before: 267")
print(f"Features after: {len(selected_features)}")

In [ ]:
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=0.1,
    random_state=42,
    max_iter=1000,

)

lr.fit(X_train, y_train)

KS statistic — maximum separation between defaulters and non-defaulters

Cumulative % of each group as we move from lowest to highest predicted default probability

KS = maximum vertical gap between the two curves. Bigger gap = better model separation.

Gini coefficient — how far your model is from random

Cumulative % of actual defaulters captured as we move through customers ranked by predicted risk

Gini = area between your model curve and the random diagonal, as a ratio. Higher = better.

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

y_pred_prob = lr.predict_proba(X_test)[:, 1]

# AUC
auc = roc_auc_score(y_test, y_pred_prob)
gini = 2 * auc - 1

# KS Statistic
from scipy.stats import ks_2samp
defaults = y_pred_prob[y_test == 1]
non_defaults = y_pred_prob[y_test == 0]
ks_stat, _ = ks_2samp(defaults, non_defaults)

print(f"AUC:   {auc:.4f}")
print(f"Gini:  {gini:.4f}")
print(f"KS:    {ks_stat:.4f}")

In [ ]:
from sklearn.metrics import classification_report

# Lower threshold to catch more defaulters
y_pred = (lr.predict_proba(X_test)[:, 1] >= 0.3).astype(int)

print(classification_report(y_test, y_pred))

Default threshold of 0.5 was adjusted to 0.3 to account for class imbalance (92% non-default vs 8% default), improving recall from 0.01 to 0.11

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Get predicted probabilities
y_prob = lr.predict_proba(X_test)[:, 1]

# Sort by probability
df_ks = pd.DataFrame({'y_test': y_test, 'y_prob': y_prob})
df_ks = df_ks.sort_values('y_prob', ascending=True).reset_index(drop=True)

# Cumulative distributions
df_ks['cum_good'] = (df_ks['y_test'] == 0).cumsum() / (df_ks['y_test'] == 0).sum()
df_ks['cum_bad'] = (df_ks['y_test'] == 1).cumsum() / (df_ks['y_test'] == 1).sum()

# KS point
ks_score = (df_ks['cum_bad'] - df_ks['cum_good']).max()

# Plot
plt.figure(figsize=(8, 5))
plt.plot(df_ks['cum_good'].values, label='Good (Non-Default)', color='blue')
plt.plot(df_ks['cum_bad'].values, label='Bad (Default)', color='red')
plt.title(f'KS Plot — KS Statistic = {ks_score:.4f}')
plt.xlabel('Percentage of Customers')
plt.ylabel('Cumulative Distribution')
plt.legend()
plt.tight_layout()
plt.savefig('ks_plot.png', dpi=150)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)
auc = roc_auc_score(y_test, y_pred_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'Logistic Regression (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random Model')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test, y_pred_prob, n_bins=7
)

plt.figure(figsize=(8, 6))
plt.plot(mean_predicted_value, fraction_of_positives,
         marker='o', label='Logistic Regression')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfect calibration')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of actual defaulters')
plt.title('Calibration Plot')
plt.legend()
plt.show()

In [ ]:
# PDO Scaling parameters
PDO = 20
base_score = 600
base_odds = 50

In [ ]:
# Calculate scaling factors
factor = PDO / np.log(2)
offset = base_score - (factor * np.log(base_odds))

In [ ]:
# Convert predicted probabilities to scores
y_pred_prob = lr.predict_proba(X_test)[:, 1]

log_odds = np.log((1 - y_pred_prob) / y_pred_prob)

scores = offset + factor * log_odds
scores = scores.round().astype(int)

print(f"Min score: {scores.min()}")
print(f"Max score: {scores.max()}")
print(f"Mean score: {scores.mean():.0f}")

In [ ]:
# Score band distribution
import pandas as pd

score_df = pd.DataFrame({
    'score': scores,
    'actual': y_test.values
})

bins = [0, 500, 520, 540, 560, 580, 600, 620, 640, 700]
labels = ['<500', '500-520', '520-540', '540-560', '560-580', '580-600', '600-620', '620-640', '640+']

score_df['score_band'] = pd.cut(score_df['score'], bins=bins, labels=labels)

band_summary = score_df.groupby('score_band', observed=True).agg(
    total_customers=('actual', 'count'),
    total_defaults=('actual', 'sum')
).reset_index()

band_summary['default_rate'] = (band_summary['total_defaults'] / band_summary['total_customers'] * 100).round(1)
band_summary['default_rate'] = band_summary['default_rate'].astype(str) + '%'

print(band_summary.to_string(index=False))

In [ ]:
band_summary_plot = band_summary.copy()
band_summary_plot['default_rate'] = band_summary_plot['default_rate'].str.replace('%', '').astype(float)

In [ ]:
# Customer level score output
score_output = pd.DataFrame({
    'SK_ID_CURR': final_df.loc[y_test.index, 'SK_ID_CURR'].values,
    'TARGET': y_test.values,
    'DEFAULT_PROBABILITY': y_pred_prob.round(4),
    'CREDIT_SCORE': scores
})

score_output = score_output.sort_values('CREDIT_SCORE', ascending=False).reset_index(drop=True)

print(score_output.head(10))

In [ ]:
# Scorecard points per feature
coefficients = pd.Series(lr.coef_[0], index=X_train.columns)
intercept = lr.intercept_[0]

scorecard = pd.DataFrame({
    'feature': coefficients.index,
    'coefficient': coefficients.values,
    'points': -(coefficients.values * factor).round(0).astype(int)
})

scorecard = scorecard.sort_values('points', ascending=False).reset_index(drop=True)

print(f"Intercept points: {-(intercept * factor + offset).round(0).astype(int)}")
print()
print(scorecard.to_string(index=False))

In [ ]:
band_summary_plot = band_summary.copy()
band_summary_plot['default_rate'] = band_summary_plot['default_rate'].str.replace('%', '').astype(float)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histogram — score distribution by default status
defaults = score_output[score_output['TARGET'] == 1]['CREDIT_SCORE']
non_defaults = score_output[score_output['TARGET'] == 0]['CREDIT_SCORE']

axes[0].hist(non_defaults, bins=20, color='green', alpha=0.6, label='Non-Default')
axes[0].hist(defaults, bins=20, color='red', alpha=0.6, label='Default')
axes[0].axvline(x=540, color='black', linestyle='--', label='Cutoff 540')
axes[0].set_title('Score Distribution by Default Status')
axes[0].set_xlabel('Credit Score')
axes[0].set_ylabel('Number of Customers')
axes[0].legend()

# Pie chart — overall split
status_counts = score_output['TARGET'].value_counts()
axes[1].pie(status_counts, labels=['Non-Default', 'Default'],
            colors=['green', 'red'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Default vs Non-Default Distribution')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

bar_colors = []
for rate in band_summary_plot['default_rate']:
    if rate >= 20:
        bar_colors.append('red')
    elif rate >= 5:
        bar_colors.append('yellow')
    else:
        bar_colors.append('green')

plt.bar(band_summary_plot['score_band'].astype(str),
        band_summary_plot['default_rate'],
        color=bar_colors, edgecolor='black')
plt.title('Default Rate by Score Band')
plt.xlabel('Score Band')
plt.ylabel('Default Rate (%)')
plt.xticks(rotation=45)
for i, v in enumerate(band_summary_plot['default_rate']):
    plt.text(i, v + 0.3, f'{v}%', ha='center', fontsize=9)
plt.tight_layout()
plt.show()